## Task 1: Building a Fault-Tolerant Multi-Source ETL Pipeline

In [54]:
!pip install pandas matplotlib requests mysql-connector-python python-dotenv

### 1. Extract Data from JSONPlaceholder API

First, we'll extract data from the JSONPlaceholder API, specifically from the `/users` and `/posts` endpoints. We'll implement robust error handling, including retries and timeouts, to ensure fault tolerance. After fetching the data, we'll normalize any nested JSON fields using `pd.json_normalize()` to flatten them into a DataFrame.

In [55]:
import requests
import pandas as pd
import time
from requests.exceptions import ConnectionError, Timeout, HTTPError, RequestException

# Base URL for JSONPlaceholder API
BASE_URL = 'https://jsonplaceholder.typicode.com'

def fetch_data_with_retry(url, retries=5, timeout=10):
    """Fetches data from a given URL with retries and timeout."""
    for i in range(retries):
        try:
            response = requests.get(url, timeout=timeout)
            response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
            return response.json()
        except Timeout:
            print(f"Request timed out for {url}. Retrying ({i + 1}/{retries})...")
        except ConnectionError:
            print(f"Connection error for {url}. Retrying ({i + 1}/{retries})...")
        except HTTPError as e:
            print(f"HTTP error {e.response.status_code} for {url}. Retrying ({i + 1}/{retries})...")
            if e.response.status_code == 404: # If not found, no point in retrying
                return None
        except RequestException as e:
            print(f"An error occurred: {e}. Retrying ({i + 1}/{retries})...")
        time.sleep(2**i)  # Exponential back-off
    print(f"Failed to fetch data from {url} after {retries} retries.")
    return None

In [56]:
# Fetch users data
print("Fetching users data...")
users_data = fetch_data_with_retry(f'{BASE_URL}/users')
if users_data:
    # Normalize users data
    users_df = pd.json_normalize(users_data)
    print("Users data fetched and normalized.")
    display(users_df.head())
    display(users_df.shape)
else:
    print("Could not fetch users data.")
    users_df = pd.DataFrame()

Fetching users data...
Users data fetched and normalized.


,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution,revolutionize end-to-end systems


(10, 15)

In [57]:
# Fetch posts data
print("Fetching posts data...")
posts_data = fetch_data_with_retry(f'{BASE_URL}/posts')

if posts_data:
    posts_df = pd.DataFrame(posts_data)
    print("Posts data fetched.")
    display(posts_df.head())
else:
    print("Could not fetch posts data.")
    posts_df = pd.DataFrame()

Fetching posts data...
Posts data fetched.


,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...


### 2. Extract Data from Local Messy CSV File

Now, we'll simulate the extraction of data from a locally generated messy CSV file. This data will include some overlapping information with the API data, which we'll use for conflict resolution.

In [58]:
import io

# Simulate a messy CSV file content
csv_data = '''id,name,email,status,notes
1,Leanne Graham,Sincere@april.biz,active,Premium user
2,Ervin Howell,shanna@melissa.tv,inactive,account suspended
101,New User One,new.user@example.com,active,First local user
3,clementine bauch,Nathan@yesenia.net,active,Frequent poster
102,Another Local,another.local@example.com,pending,Needs verification
'''

# Read the CSV data into a pandas DataFrame
local_df = pd.read_csv(io.StringIO(csv_data))

print("Local CSV data extracted:")
display(local_df.head())

# Preview initial data types to identify potential issues
print("\nInitial data types for local_df:")
print(local_df.dtypes)

Local CSV data extracted:


,id,name,email,status,notes
0,1,Leanne Graham,Sincere@april.biz,active,Premium user
1,2,Ervin Howell,shanna@melissa.tv,inactive,account suspended
2,101,New User One,new.user@example.com,active,First local user
3,3,clementine bauch,Nathan@yesenia.net,active,Frequent poster
4,102,Another Local,another.local@example.com,pending,Needs verification



Initial data types for local_df:
id         int64
name      object
email     object
status    object
notes     object
dtype: object


### 3. Merge DataFrames with Conflict Resolution

We will merge `users_df` (from API) and `local_df` (from CSV) on the `email` field. A crucial part of this step is handling conflicts, particularly when the `name` field differs for the same email address across both sources.

**Conflict Resolution Strategy:**

When `email` addresses match but `name` values differ, we will prioritize the `name` from the **API data (`users_df`)**. The rationale is that the API is often considered the primary and most authoritative source for core user information. Other fields from the `local_df` (like `status`, `notes`) that are not present in `users_df` will be added without conflict.

In [59]:
# Prepare data for merging: standardize email to lowercase and select relevant columns

# For users_df (API data), select key columns and rename some for clarity/consistency
users_for_merge = users_df[['id', 'name', 'email']].copy()
users_for_merge['email'] = users_for_merge['email'].str.lower()
users_for_merge.rename(columns={'id': 'user_api_id', 'name': 'name_api'}, inplace=True)

# For local_df (CSV data), select key columns and rename some
local_for_merge = local_df[['id', 'name', 'email', 'status', 'notes']].copy()
local_for_merge['email'] = local_for_merge['email'].str.lower()
local_for_merge.rename(columns={'id': 'user_local_id', 'name': 'name_local'}, inplace=True)

print("Prepared API Users Data for Merge:")
display(users_for_merge.head())
print("\nPrepared Local Data for Merge:")
display(local_for_merge.head())

Prepared API Users Data for Merge:


,user_api_id,name_api,email
0,1,Leanne Graham,sincere@april.biz
1,2,Ervin Howell,shanna@melissa.tv
2,3,Clementine Bauch,nathan@yesenia.net
3,4,Patricia Lebsack,julianne.oconner@kory.org
4,5,Chelsey Dietrich,lucio_hettinger@annie.ca



Prepared Local Data for Merge:


,user_local_id,name_local,email,status,notes
0,1,Leanne Graham,sincere@april.biz,active,Premium user
1,2,Ervin Howell,shanna@melissa.tv,inactive,account suspended
2,101,New User One,new.user@example.com,active,First local user
3,3,clementine bauch,nathan@yesenia.net,active,Frequent poster
4,102,Another Local,another.local@example.com,pending,Needs verification


In [60]:
# Perform the merge
# We'll do a full outer merge to keep all records from both sources initially
merged_df = pd.merge(
    users_for_merge,
    local_for_merge,
    on='email',
    how='outer',
    suffixes=('_api', '_local')
)

# Apply conflict resolution for 'name': prioritize API name
merged_df['name'] = merged_df['name_api'].fillna(merged_df['name_local'])

# Drop original name columns used for resolution and rename user_api_id to id for consistency if desired
merged_df.drop(columns=['name_api', 'name_local'], inplace=True)

# Handle the user_api_id and user_local_id: if an email exists in API, use its ID; otherwise, use local ID (or create a new one later)
# For simplicity, if user_api_id is null, it means it's a new user from local_df. We can use user_local_id for these.
# For now, let's keep both and decide on a final 'id' column later if needed, or if we ensure no duplicates on 'id' itself.
# For now, we'll keep 'user_api_id' as the primary identifier if available.
merged_df.rename(columns={'user_api_id': 'id'}, inplace=True)

# Fill 'id' with 'user_local_id' where 'id' (original user_api_id) is null, to give new IDs to local-only users.
merged_df['id'] = merged_df['id'].fillna(merged_df['user_local_id'])

# Drop the user_local_id as it's now incorporated or redundant for merged records
merged_df.drop(columns=['user_local_id'], inplace=True)

# Display the merged DataFrame
print("Merged DataFrame with conflict resolution:")
display(merged_df.head())
print(f"Total records after merge: {len(merged_df)}")

# Also show some conflicts if any
print("\nRecords where name was conflicting (API name chosen):")
# To see where API name was chosen over a differing local name for existing emails
conflicting_emails = pd.merge(
    users_for_merge[['email', 'name_api']],
    local_for_merge[['email', 'name_local']],
    on='email',
    how='inner'
)
conflicting_emails = conflicting_emails[conflicting_emails['name_api'].str.lower() != conflicting_emails['name_local'].str.lower()]

if not conflicting_emails.empty:
    display(conflicting_emails)
else:
    print("No explicit name conflicts found for matching emails.")

Merged DataFrame with conflict resolution:


,id,email,status,notes,name
0,102.0,another.local@example.com,pending,Needs verification,Another Local
1,9.0,chaim_mcdermott@dana.io,NaN,NaN,Glenna Reichert
2,4.0,julianne.oconner@kory.org,NaN,NaN,Patricia Lebsack
3,6.0,karley_dach@jasper.info,NaN,NaN,Mrs. Dennis Schulist
4,5.0,lucio_hettinger@annie.ca,NaN,NaN,Chelsey Dietrich


Total records after merge: 12

Records where name was conflicting (API name chosen):
No explicit name conflicts found for matching emails.


### 4. Apply Data Cleaning Techniques

Now, we'll systematically apply all six cleaning techniques to the `merged_df`:

1.  **Nulls:** Identify and handle missing values.
2.  **Duplicates:** Remove duplicate rows.
3.  **Casing:** Standardize text casing.
4.  **Types:** Ensure correct data types.
5.  **Whitespace:** Remove leading/trailing whitespace.
6.  **Outliers:** Identify and address outliers (if applicable to the current dataset).

#### 4.1. Handle Null Values

We'll check for null values and decide on an imputation or removal strategy. For this dataset, we will fill `status` with 'unknown' for users that came only from the API, and `notes` with 'N/A' as they are not critical fields.

In [61]:
print("Null values before handling:")
print(merged_df.isnull().sum())

# Fill nulls in 'status' with 'unknown' for users from API that don't have local status
merged_df['status'].fillna('unknown', inplace=True)

# Fill nulls in 'notes' with 'N/A'
merged_df['notes'].fillna('N/A', inplace=True)

# For the 'id' column, if there are still any nulls (e.g., if neither user_api_id nor user_local_id existed which is unlikely after merge logic)
# we should address them, but based on current merge logic, 'id' should be filled.
# Let's ensure 'id' is an integer, so we can convert it after filling. It might be float due to fillna initially.
merged_df['id'] = merged_df['id'].astype(int)

print("\nNull values after handling:")
print(merged_df.isnull().sum())

print("\nDataFrame after null handling (head):")
display(merged_df.head())

Null values before handling:
id        0
email     0
status    7
notes     7
name      0
dtype: int64

Null values after handling:
id        0
email     0
status    0
notes     0
name      0
dtype: int64

DataFrame after null handling (head):


/tmp/ipykernel_4494/3319975909.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df['status'].fillna('unknown', inplace=True)
/tmp/ipykernel_4494/3319975909.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', t

,id,email,status,notes,name
0,102,another.local@example.com,pending,Needs verification,Another Local
1,9,chaim_mcdermott@dana.io,unknown,N/A,Glenna Reichert
2,4,julianne.oconner@kory.org,unknown,N/A,Patricia Lebsack
3,6,karley_dach@jasper.info,unknown,N/A,Mrs. Dennis Schulist
4,5,lucio_hettinger@annie.ca,unknown,N/A,Chelsey Dietrich


#### 4.2. Handle Duplicates

We will identify and remove duplicate rows based on the 'id' column, assuming 'id' uniquely identifies a user after our merge logic has resolved email conflicts. We will keep the first occurrence.

In [62]:
print(f"Number of rows before duplicate removal: {len(merged_df)}")

# Identify duplicates based on the 'id' column
duplicates = merged_df[merged_df.duplicated(subset=['id'], keep=False)]

if not duplicates.empty:
    print("\nIdentified duplicate rows (based on 'id'):")
    display(duplicates.sort_values(by='id'))
    merged_df.drop_duplicates(subset=['id'], keep='first', inplace=True)
    print(f"Number of rows after duplicate removal: {len(merged_df)}")
else:
    print("\nNo duplicate rows found based on 'id'.")

print("\nDataFrame after duplicate handling (head):")
display(merged_df.head())

Number of rows before duplicate removal: 12

No duplicate rows found based on 'id'.

DataFrame after duplicate handling (head):


,id,email,status,notes,name
0,102,another.local@example.com,pending,Needs verification,Another Local
1,9,chaim_mcdermott@dana.io,unknown,N/A,Glenna Reichert
2,4,julianne.oconner@kory.org,unknown,N/A,Patricia Lebsack
3,6,karley_dach@jasper.info,unknown,N/A,Mrs. Dennis Schulist
4,5,lucio_hettinger@annie.ca,unknown,N/A,Chelsey Dietrich


#### 4.3. Standardize Casing

We'll standardize string columns (like 'name', 'email', 'status', 'notes') to a consistent casing (e.g., title case for names, lowercase for emails, lowercase for status and notes for easier comparison).

In [63]:
# Standardize 'name' to title case
merged_df['name'] = merged_df['name'].str.title()

# Standardize 'email' and 'status' to lowercase (already done for email during merge prep, but good to re-apply)
merged_df['email'] = merged_df['email'].str.lower()
merged_df['status'] = merged_df['status'].str.lower()

# Standardize 'notes' to lowercase for consistency
merged_df['notes'] = merged_df['notes'].str.lower()

print("DataFrame after casing standardization (head):")
display(merged_df.head())

DataFrame after casing standardization (head):


,id,email,status,notes,name
0,102,another.local@example.com,pending,needs verification,Another Local
1,9,chaim_mcdermott@dana.io,unknown,n/a,Glenna Reichert
2,4,julianne.oconner@kory.org,unknown,n/a,Patricia Lebsack
3,6,karley_dach@jasper.info,unknown,n/a,Mrs. Dennis Schulist
4,5,lucio_hettinger@annie.ca,unknown,n/a,Chelsey Dietrich


#### 4.4. Ensure Correct Data Types

We'll verify and enforce appropriate data types for each column to optimize memory usage and ensure correct operations. We've already handled `id` as `int`.

In [64]:
print("Data types before type conversion:")
print(merged_df.dtypes)

# Ensure 'id' is integer (already done after null handling, confirming)
merged_df['id'] = merged_df['id'].astype(int)

# All other columns are currently objects (strings), which is appropriate for their content.
# No further specific type conversions are strictly necessary beyond ensuring 'id' is int.

print("\nData types after type conversion:")
print(merged_df.dtypes)

print("\nDataFrame after type handling (head):")
display(merged_df.head())

Data types before type conversion:
id         int64
email     object
status    object
notes     object
name      object
dtype: object

Data types after type conversion:
id         int64
email     object
status    object
notes     object
name      object
dtype: object

DataFrame after type handling (head):


,id,email,status,notes,name
0,102,another.local@example.com,pending,needs verification,Another Local
1,9,chaim_mcdermott@dana.io,unknown,n/a,Glenna Reichert
2,4,julianne.oconner@kory.org,unknown,n/a,Patricia Lebsack
3,6,karley_dach@jasper.info,unknown,n/a,Mrs. Dennis Schulist
4,5,lucio_hettinger@annie.ca,unknown,n/a,Chelsey Dietrich


#### 4.5. Remove Whitespace

Remove leading/trailing whitespace from all string columns.

In [65]:
for col in ['name', 'email', 'status', 'notes']:
    if col in merged_df.columns and merged_df[col].dtype == 'object':
        merged_df[col] = merged_df[col].str.strip()

print("DataFrame after whitespace removal (head):")
display(merged_df.head())

DataFrame after whitespace removal (head):


,id,email,status,notes,name
0,102,another.local@example.com,pending,needs verification,Another Local
1,9,chaim_mcdermott@dana.io,unknown,n/a,Glenna Reichert
2,4,julianne.oconner@kory.org,unknown,n/a,Patricia Lebsack
3,6,karley_dach@jasper.info,unknown,n/a,Mrs. Dennis Schulist
4,5,lucio_hettinger@annie.ca,unknown,n/a,Chelsey Dietrich


#### 4.6. Identify and Address Outliers

For this particular dataset (user information with name, email, status, notes), numerical outliers are not directly applicable. If we had quantitative data (e.g., transaction amounts, activity scores), we would apply statistical methods (e.g., Z-score, IQR) or domain-specific rules to identify and handle outliers. For now, we will note this step as addressed by acknowledging its irrelevance for the current column types.

#### 4.3. Standardize Casing

We'll standardize string columns (like 'name', 'email', 'status', 'notes') to a consistent casing (e.g., title case for names, lowercase for emails, lowercase for status and notes for easier comparison).

In [66]:
# Standardize 'name' to title case
merged_df['name'] = merged_df['name'].str.title()

# Standardize 'email' and 'status' to lowercase (already done for email during merge prep, but good to re-apply)
merged_df['email'] = merged_df['email'].str.lower()
merged_df['status'] = merged_df['status'].str.lower()

# Standardize 'notes' to lowercase for consistency
merged_df['notes'] = merged_df['notes'].str.lower()

print("DataFrame after casing standardization (head):")
display(merged_df.head())

DataFrame after casing standardization (head):


,id,email,status,notes,name
0,102,another.local@example.com,pending,needs verification,Another Local
1,9,chaim_mcdermott@dana.io,unknown,n/a,Glenna Reichert
2,4,julianne.oconner@kory.org,unknown,n/a,Patricia Lebsack
3,6,karley_dach@jasper.info,unknown,n/a,Mrs. Dennis Schulist
4,5,lucio_hettinger@annie.ca,unknown,n/a,Chelsey Dietrich


#### 4.4. Ensure Correct Data Types

We'll verify and enforce appropriate data types for each column to optimize memory usage and ensure correct operations. We've already handled `id` as `int`.

In [67]:
print("Data types before type conversion:")
print(merged_df.dtypes)

# Ensure 'id' is integer (already done after null handling, confirming)
merged_df['id'] = merged_df['id'].astype(int)

# All other columns are currently objects (strings), which is appropriate for their content.
# No further specific type conversions are strictly necessary beyond ensuring 'id' is int.

print("\nData types after type conversion:")
print(merged_df.dtypes)

print("\nDataFrame after type handling (head):")
display(merged_df.head())

Data types before type conversion:
id         int64
email     object
status    object
notes     object
name      object
dtype: object

Data types after type conversion:
id         int64
email     object
status    object
notes     object
name      object
dtype: object

DataFrame after type handling (head):


,id,email,status,notes,name
0,102,another.local@example.com,pending,needs verification,Another Local
1,9,chaim_mcdermott@dana.io,unknown,n/a,Glenna Reichert
2,4,julianne.oconner@kory.org,unknown,n/a,Patricia Lebsack
3,6,karley_dach@jasper.info,unknown,n/a,Mrs. Dennis Schulist
4,5,lucio_hettinger@annie.ca,unknown,n/a,Chelsey Dietrich


#### 4.5. Remove Whitespace

Remove leading/trailing whitespace from all string columns.

In [68]:
for col in ['name', 'email', 'status', 'notes']:
    if col in merged_df.columns and merged_df[col].dtype == 'object':
        merged_df[col] = merged_df[col].str.strip()

print("DataFrame after whitespace removal (head):")
display(merged_df.head())

DataFrame after whitespace removal (head):


,id,email,status,notes,name
0,102,another.local@example.com,pending,needs verification,Another Local
1,9,chaim_mcdermott@dana.io,unknown,n/a,Glenna Reichert
2,4,julianne.oconner@kory.org,unknown,n/a,Patricia Lebsack
3,6,karley_dach@jasper.info,unknown,n/a,Mrs. Dennis Schulist
4,5,lucio_hettinger@annie.ca,unknown,n/a,Chelsey Dietrich


#### 4.6. Identify and Address Outliers

For this particular dataset (user information with name, email, status, notes), numerical outliers are not directly applicable. If we had quantitative data (e.g., transaction amounts, activity scores), we would apply statistical methods (e.g., Z-score, IQR) or domain-specific rules to identify and handle outliers. For now, we will note this step as addressed by acknowledging its irrelevance for the current column types.

### 5. Load the Final Unified Dataset to MySql and CSV

Finally, we will load our cleaned and unified `merged_df` into two destinations:

1.  **MySql Database:** Ensuring no duplicate rows are inserted on a second run.
2.  **CSV File:** A simple export to a local CSV file.

#### 5.1. Load to MySQL

We will connect to a MySQL database and create a table. To prevent duplicate insertions on a second run, we'll define a primary key (`id`) and a unique key (`email`), and use `INSERT IGNORE INTO` for data insertion. This ensures that existing records (identified by `id` or `email`) are not re-inserted.

In [73]:
import mysql.connector
from dotenv import load_dotenv
import os
import numpy as np

# Load environment variables from .env file
load_dotenv()

# MySQL connection details from .env
DB_HOST = os.getenv("DB_HOST")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")
TABLE_NAME = 'users_data'

# Debugging: Print DB_HOST to verify it's loaded correctly
print(f"DEBUG: DB_HOST loaded: {DB_HOST}")

# Initialize conn and cursor to None
conn = None
cursor = None

try:
    # --- Step 1: Connect to MySQL server (without specifying a database) to ensure DB_NAME exists ---
    print("Attempting to connect to MySQL server to ensure database exists...")
    conn_no_db = mysql.connector.connect(
        host=DB_HOST,
        user=DB_USER,
        password=DB_PASSWORD
    )
    cursor_no_db = conn_no_db.cursor()

    # Create the database if it does not exist
    cursor_no_db.execute(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
    print(f"Database '{DB_NAME}' ensured to exist.")

    cursor_no_db.close()
    conn_no_db.close()
    print("Initial connection to server closed.")

    # --- Step 2: Connect to the specific database for table and data operations ---
    print(f"Attempting to connect to database '{DB_NAME}'...")
    conn = mysql.connector.connect(
        host=DB_HOST,
        user=DB_USER,
        password=DB_PASSWORD,
        database=DB_NAME
    )
    if conn.is_connected():
        cursor = conn.cursor()
        print(f"Successfully connected to MySQL database: {DB_NAME}")

        # --- Table Creation/Check ---
        print(f"Checking if table '{TABLE_NAME}' exists in '{DB_NAME}'...")
        cursor.execute(f"SHOW TABLES LIKE '{TABLE_NAME}'")
        table_exists = cursor.fetchone()

        if table_exists:
            print(f"Table '{TABLE_NAME}' already exists.")
            # DESCRIBE table to check its schema
            cursor.execute(f"DESCRIBE {TABLE_NAME}")
            table_description = cursor.fetchall()
            print(f"Current schema of '{TABLE_NAME}':")
            for col in table_description:
                print(f"  {col[0]} ({col[1]})") # Print column name and type
        else:
            print(f"Table '{TABLE_NAME}' does not exist. Attempting to create...")
            create_table_sql = f'''
            CREATE TABLE {TABLE_NAME} (
                id INT PRIMARY KEY,
                email VARCHAR(255) UNIQUE,
                status VARCHAR(50),
                notes TEXT,
                name VARCHAR(255)
            );
            '''
            cursor.execute(create_table_sql)
            conn.commit()
            print(f"Table '{TABLE_NAME}' successfully created.")
            # After creating, describe it to confirm
            cursor.execute(f"DESCRIBE {TABLE_NAME}")
            table_description = cursor.fetchall()
            print(f"Schema of newly created '{TABLE_NAME}':")
            for col in table_description:
                print(f"  {col[0]} ({col[1]}) মুঠ") # Print column name and type

        # --- Data Insertion ---
        print(f"Preparing to insert/update data into table '{TABLE_NAME}'...")

        # Convert NaN values in the DataFrame to None for MySQL compatibility
        # This prevents MySQL from interpreting 'nan' as a column name or a faulty SELECT
        merged_df_for_insert = merged_df[['id', 'email', 'status', 'notes', 'name']].copy()
        merged_df_for_insert = merged_df_for_insert.replace({np.nan: None})

        # We'll convert DataFrame to a list of tuples for efficient insertion
        data_to_insert = merged_df_for_insert.values.tolist()

        # SQL INSERT IGNORE INTO statement to prevent duplicate rows based on primary key (id) and unique key (email)
        insert_sql = f"INSERT IGNORE INTO {TABLE_NAME} (id, email, status, notes, name) VALUES (%s, %s, %s, %s, %s)"

        cursor.executemany(insert_sql, data_to_insert)
        conn.commit()

        print(f"Data insertion complete. Duplicates (based on id/email) were ignored.")

        # --- Data Verification ---
        print(f"Verifying data in MySQL table '{TABLE_NAME}'...")
        cursor.execute(f"SELECT id, email, status, notes, name FROM {TABLE_NAME}")
        fetched_data = cursor.fetchall()
        # Get column names from cursor description
        column_names = [i[0] for i in cursor.description]
        verification_df = pd.DataFrame(fetched_data, columns=column_names)

        print("\nData in MySQL table (first 5 rows):")
        display(verification_df.head())
        print(f"Total records in MySQL: {len(verification_df)}")

except mysql.connector.Error as err:
    print(f"Error connecting to MySQL: {err}")
finally:
    if cursor is not None:
        cursor.close()
    if conn is not None and conn.is_connected():
        conn.close()
        print("MySQL connection closed.")
    elif conn is not None and not conn.is_connected():
        print("MySQL connection was not successfully established or was already closed.")

DEBUG: DB_HOST loaded: None
Attempting to connect to MySQL server to ensure database exists...
Error connecting to MySQL: 2003: Can't connect to MySQL server on 'None:3306' (Errno 111: Connection refused)


#### 5.2. Load to CSV

Export the final unified dataset to a CSV file.

In [70]:
OUTPUT_CSV_FILE = 'unified_users_data.csv'

merged_df.to_csv(OUTPUT_CSV_FILE, index=False)

print(f"Final unified dataset loaded to '{OUTPUT_CSV_FILE}'.")

Final unified dataset loaded to 'unified_users_data.csv'.
